# Health-Focused Retrofit Prioritisation in England

**Linking housing energy efficiency, fuel poverty and respiratory health outcomes**

> This notebook generates sample data and produces all 5 portfolio visualisations.
> Replace the sample data section with real downloads from EPC, DESNZ and OHID Fingertips.

**Disclaimer:** This analysis identifies overlapping risk patterns. It does not prove causation between housing quality and health outcomes.

In [ ]:
# Install dependencies (if needed in Colab)
# !pip install pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
print('Libraries loaded successfully.')

In [ ]:
# ================================================================
# SAMPLE DATA — 20 realistic English local authorities
# Replace this cell with: df = pd.read_csv('data/output/health_focused_retrofit_prioritisation_england.csv')
# ================================================================

np.random.seed(42)

local_authorities = [
    ('E07000223', 'Adur', 'South East'), ('E07000026', 'Allerdale', 'North West'),
    ('E07000032', 'Amber Valley', 'East Midlands'), ('E06000022', 'Bath and NE Somerset', 'South West'),
    ('E06000055', 'Bedford', 'East'), ('E08000025', 'Birmingham', 'West Midlands'),
    ('E06000008', 'Blackburn with Darwen', 'North West'), ('E06000009', 'Blackpool', 'North West'),
    ('E07000033', 'Bolsover', 'East Midlands'), ('E08000001', 'Bolton', 'North West'),
    ('E06000058', 'Bournemouth CP', 'South West'), ('E06000036', 'Bracknell Forest', 'South East'),
    ('E08000032', 'Bradford', 'Yorkshire and The Humber'), ('E07000067', 'Braintree', 'East'),
    ('E07000143', 'Breckland', 'East'), ('E09000005', 'Brent', 'London'),
    ('E07000068', 'Brentwood', 'East'), ('E06000043', 'Brighton and Hove', 'South East'),
    ('E06000023', 'Bristol', 'South West'), ('E07000144', 'Broadland', 'East')
]

rows = []
for code, name, region in local_authorities:
    pct_below_c = np.random.uniform(25, 75)
    fp_rate = np.random.uniform(5, 22)
    copd = np.random.uniform(80, 350)
    asthma = np.random.uniform(40, 180)
    resp_mort = np.random.uniform(60, 200)
    avg_epc = np.random.uniform(50, 75)
    co2 = np.random.uniform(2.5, 6.0)
    resp_risk = (copd + asthma + resp_mort) / 3
    rows.append({
        'local_authority_code': code,
        'local_authority_name': name,
        'region': region,
        'avg_epc_score': round(avg_epc, 1),
        'percent_homes_epc_c_or_above': round(100 - pct_below_c, 1),
        'percent_homes_below_epc_c': round(pct_below_c, 1),
        'avg_co2_emissions_per_property': round(co2, 2),
        'fuel_poverty_rate': round(fp_rate, 1),
        'fuel_poor_households': int(np.random.uniform(1000, 25000)),
        'copd_admission_rate': round(copd, 1),
        'asthma_admission_rate': round(asthma, 1),
        'respiratory_mortality_rate': round(resp_mort, 1),
        'respiratory_health_risk': round(resp_risk, 1)
    })

df = pd.DataFrame(rows)

# Normalise and score
def normalise(s): return (s - s.min()) / (s.max() - s.min())

df['norm_epc'] = normalise(df['percent_homes_below_epc_c'])
df['norm_fp'] = normalise(df['fuel_poverty_rate'])
df['norm_health'] = normalise(df['respiratory_health_risk'])

df['retrofit_health_priority_score'] = (
    0.4 * df['norm_epc'] + 0.3 * df['norm_fp'] + 0.3 * df['norm_health']
).round(4)

p20, p40, p80 = df['retrofit_health_priority_score'].quantile([0.2, 0.4, 0.8])
def band(s):
    if s >= p80: return 'Very High'
    elif s >= p40: return 'High'
    elif s >= p20: return 'Medium'
    else: return 'Low'

df['priority_band'] = df['retrofit_health_priority_score'].apply(band)
df = df.sort_values('retrofit_health_priority_score', ascending=False).reset_index(drop=True)
print(f'Dataset ready: {len(df)} local authorities')
print(df[['local_authority_name','priority_band','retrofit_health_priority_score']].head(10).to_string())


In [ ]:
# ================================================================
# VISUAL 1: Top 20 Local Authorities for Health-Focused Retrofit
# ================================================================

top20 = df.head(20).sort_values('retrofit_health_priority_score')

colour_map = {'Very High': '#d73027', 'High': '#f46d43', 'Medium': '#74add1', 'Low': '#4575b4'}
colours = [colour_map.get(b, '#999') for b in top20['priority_band']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top20['local_authority_name'], top20['retrofit_health_priority_score'],
               color=colours, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Retrofit-Health Priority Score', fontsize=12)
ax.set_title('Top 20 Local Authorities for Health-Focused Retrofit\n'
             'Combined housing inefficiency, fuel poverty and respiratory health risk',
             fontsize=13, fontweight='bold', pad=15)
ax.axvline(x=df['retrofit_health_priority_score'].mean(), color='#555', linestyle='--',
           linewidth=1.2, alpha=0.7, label=f'Mean score ({df["retrofit_health_priority_score"].mean():.3f})')
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in colour_map.items()]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0],color='#555',linestyle='--',label='Mean')],
          loc='lower right', fontsize=9)
ax.text(0.98, 0.02, 'Note: score reflects overlap of housing, fuel poverty and health risk.\nDoes not prove causation.',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='#666', style='italic')
plt.tight_layout()
plt.savefig('visual1_top20_priority_las.png', bbox_inches='tight')
plt.show()
print('Visual 1 saved.')

In [ ]:
# ================================================================
# VISUAL 2: Housing Inefficiency vs Fuel Poverty
# ================================================================

fig, ax = plt.subplots(figsize=(10, 7))

for band_name, colour in colour_map.items():
    subset = df[df['priority_band'] == band_name]
    ax.scatter(subset['percent_homes_below_epc_c'], subset['fuel_poverty_rate'],
               c=colour, label=band_name, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

m, b, r, p, _ = stats.linregress(df['percent_homes_below_epc_c'], df['fuel_poverty_rate'])
x_line = np.linspace(df['percent_homes_below_epc_c'].min(), df['percent_homes_below_epc_c'].max(), 100)
ax.plot(x_line, m * x_line + b, color='#333', linewidth=1.5, linestyle='--', alpha=0.6,
        label=f'Trend (r={r:.2f})')

for _, row in df.iterrows():
    ax.annotate(row['local_authority_name'], (row['percent_homes_below_epc_c'], row['fuel_poverty_rate']),
                fontsize=7, alpha=0.7, xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('Homes Below EPC Rating C (%)', fontsize=12)
ax.set_ylabel('Fuel Poverty Rate (%)', fontsize=12)
ax.set_title('Housing Inefficiency vs Fuel Poverty\nby Local Authority — Priority Band',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=9)
ax.text(0.02, 0.98, 'Association shown. Causation not claimed.',
        transform=ax.transAxes, ha='left', va='top', fontsize=8, color='#666', style='italic')
plt.tight_layout()
plt.savefig('visual2_epc_vs_fuel_poverty.png', bbox_inches='tight')
plt.show()
print('Visual 2 saved.')

In [ ]:
# ================================================================
# VISUAL 3: Fuel Poverty vs Respiratory Health Outcomes
# ================================================================

fig, ax = plt.subplots(figsize=(10, 7))

for band_name, colour in colour_map.items():
    subset = df[df['priority_band'] == band_name]
    ax.scatter(subset['fuel_poverty_rate'], subset['copd_admission_rate'],
               c=colour, label=band_name, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

m, b, r, p, _ = stats.linregress(df['fuel_poverty_rate'], df['copd_admission_rate'])
x_line = np.linspace(df['fuel_poverty_rate'].min(), df['fuel_poverty_rate'].max(), 100)
ax.plot(x_line, m * x_line + b, color='#333', linewidth=1.5, linestyle='--', alpha=0.6,
        label=f'Trend (r={r:.2f})')

for _, row in df.iterrows():
    ax.annotate(row['local_authority_name'], (row['fuel_poverty_rate'], row['copd_admission_rate']),
                fontsize=7, alpha=0.7, xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('Fuel Poverty Rate (%)', fontsize=12)
ax.set_ylabel('COPD Unplanned Admission Rate (per 100,000)', fontsize=12)
ax.set_title('Fuel Poverty vs COPD Admission Rate\nby Local Authority — Priority Band',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=9)
ax.text(0.02, 0.98, 'Association shown. Causation not claimed.',
        transform=ax.transAxes, ha='left', va='top', fontsize=8, color='#666', style='italic')
plt.tight_layout()
plt.savefig('visual3_fuel_poverty_vs_respiratory.png', bbox_inches='tight')
plt.show()
print('Visual 3 saved.')

In [ ]:
# ================================================================
# VISUAL 4: Retrofit-Health Priority Matrix (Bubble Chart)
# ================================================================

fig, ax = plt.subplots(figsize=(11, 8))

bubble_size = (df['retrofit_health_priority_score'] * 800 + 50)

for band_name, colour in colour_map.items():
    subset = df[df['priority_band'] == band_name]
    sizes = (subset['retrofit_health_priority_score'] * 800 + 50)
    ax.scatter(subset['percent_homes_below_epc_c'], subset['copd_admission_rate'],
               s=sizes, c=colour, alpha=0.7, edgecolors='white', linewidth=0.8, label=band_name)

for _, row in df.iterrows():
    ax.annotate(row['local_authority_name'],
                (row['percent_homes_below_epc_c'], row['copd_admission_rate']),
                fontsize=7.5, alpha=0.8, xytext=(4, 4), textcoords='offset points')

median_epc = df['percent_homes_below_epc_c'].median()
median_copd = df['copd_admission_rate'].median()
ax.axvline(x=median_epc, color='#888', linestyle=':', linewidth=1.2, alpha=0.7)
ax.axhline(y=median_copd, color='#888', linestyle=':', linewidth=1.2, alpha=0.7)
ax.text(median_epc + 1, ax.get_ylim()[1]*0.98, 'Median EPC\nbelow-C', fontsize=7.5, color='#888', va='top')
ax.text(ax.get_xlim()[0], median_copd + 2, 'Median COPD\nrate', fontsize=7.5, color='#888')

ax.set_xlabel('Homes Below EPC Rating C (%)', fontsize=12)
ax.set_ylabel('COPD Admission Rate (per 100,000)', fontsize=12)
ax.set_title('Retrofit-Health Priority Matrix\nBubble size = combined priority score',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(title='Priority Band', fontsize=9, title_fontsize=9)
ax.text(0.02, 0.02, 'Larger bubbles = higher combined priority.\nDoes not imply direct causal relationship.',
        transform=ax.transAxes, ha='left', va='bottom', fontsize=8, color='#666', style='italic')
plt.tight_layout()
plt.savefig('visual4_priority_matrix_bubble.png', bbox_inches='tight')
plt.show()
print('Visual 4 saved.')

In [ ]:
# ================================================================
# VISUAL 5: Priority Band Distribution
# ================================================================

band_order = ['Very High', 'High', 'Medium', 'Low']
band_counts = df['priority_band'].value_counts().reindex(band_order, fill_value=0)
band_colours = [colour_map[b] for b in band_order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

# Bar chart
bars = ax1.bar(band_order, band_counts.values, color=band_colours, edgecolor='white', linewidth=0.8)
for bar, count in zip(bars, band_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', va='bottom', fontweight='bold', fontsize=11)
ax1.set_xlabel('Priority Band', fontsize=12)
ax1.set_ylabel('Number of Local Authorities', fontsize=12)
ax1.set_title('Priority Band Distribution\n(count of local authorities)', fontsize=12, fontweight='bold')
ax1.set_ylim(0, band_counts.max() * 1.25)

# Score distribution by band (box plot)
band_data = [df[df['priority_band'] == b]['retrofit_health_priority_score'].values for b in band_order]
bp = ax2.boxplot(band_data, labels=band_order, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, colour in zip(bp['boxes'], band_colours):
    patch.set_facecolor(colour)
    patch.set_alpha(0.8)
ax2.set_xlabel('Priority Band', fontsize=12)
ax2.set_ylabel('Retrofit-Health Priority Score', fontsize=12)
ax2.set_title('Score Distribution by Priority Band', fontsize=12, fontweight='bold')

plt.suptitle('Health-Focused Retrofit Priority — Band Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visual5_priority_band_distribution.png', bbox_inches='tight')
plt.show()
print('Visual 5 saved.')
print('\nAll 5 visuals complete.')

## Next steps

1. Download real data from EPC register, DESNZ fuel poverty tables and OHID Fingertips
2. Run `src/clean_epc.py`, `src/clean_fuel_poverty.py`, `src/clean_health.py`
3. Run `src/build_dataset.py` to produce `data/output/health_focused_retrofit_prioritisation_england.csv`
4. Replace the sample data cell above with: `df = pd.read_csv('data/output/health_focused_retrofit_prioritisation_england.csv')`
5. Re-run all visualisation cells

**Remember:** This analysis identifies areas for further investigation. It does not prove causation.